# DSC 450 Project 2: Texas Weather Pull and Merge

| | |
|---|---|
| **Title** | DSC 450 Project 2: Texas Weather Pull and Merge |
| **Author** | Hali Sarah Parsons |
| **Role** | Data Wrangler |
| **Date** | 2026-10-14 |
| **Modified By** | Claude (initial draft) |
| **Description** | Pulls hourly temperature for ERCOT's four largest metros from the Open-Meteo historical weather archive, builds a population-weighted Texas temperature and degree-hour features, and joins them onto the cleaned ERCOT demand file by hour. |

**Inputs:** `eia930_erco_cleaned.csv` (from the wrangling notebook)  
**Outputs:** `weather_texas_hourly.csv` (weather only) and `eia930_erco_weather.csv` (ERCOT + weather)

### Why weather
Temperature is the strongest single driver of electricity demand: air conditioning in summer, electric heating in winter. The EIA-930 data has no weather, so the models could only learn it indirectly through season. This notebook adds it as a second data source.

**In Colab:** upload `eia930_erco_cleaned.csv` before running. No API key is needed.

## 1. Setup

In [ ]:
# Title: DSC 450 Project 2 - Texas Weather Pull and Merge
# Author: Hali Sarah Parsons
# Date: 2026-10-14
# Modified By: Claude (initial draft)
# Description: Pulls hourly Texas temperatures and merges them onto the ERCOT demand data.

import time

import numpy as np
import pandas as pd
import requests

ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'
START = '2022-01-01'
END = '2026-09-30'

## 2. Cities and weights

**Why these four metros:** Houston, Dallas-Fort Worth, San Antonio, and Austin hold most of ERCOT's load, so their temperatures track statewide demand closely.

**Why population weights:** a hot day in Dallas moves demand more than a hot day in Austin because more people live there. Weighting by metro population gives one temperature that reflects where the load is. The weights are approximate metro populations (in millions), which is accurate enough for a weighting.

In [ ]:
# WHY coordinates at city centers: the archive returns the nearest grid cell, so a center point
# represents the metro well enough.
CITIES = {
    'houston':     {'lat': 29.76, 'lon': -95.37, 'pop_m': 7.5},
    'dallas':      {'lat': 32.78, 'lon': -96.80, 'pop_m': 8.1},
    'san_antonio': {'lat': 29.42, 'lon': -98.49, 'pop_m': 2.7},
    'austin':      {'lat': 30.27, 'lon': -97.74, 'pop_m': 2.5},
}
total_pop = sum(c['pop_m'] for c in CITIES.values())
WEIGHTS = {name: c['pop_m'] / total_pop for name, c in CITIES.items()}
WEIGHTS

## 3. Pull hourly temperature

**Why UTC (`timezone=GMT`):** the ERCOT file is indexed in UTC, so pulling weather in UTC makes the join a direct match with no daylight-saving gaps or repeated hours.

**Why retry and pause:** the free API rate-limits bursts, so the pull waits between cities and retries with growing pauses. Expect it to take a minute or two.

In [ ]:
def fetch_city(name, lat, lon):
    '''Return an hourly temperature Series (°F, UTC index) for one city.

    WHY a function: the same request repeats for every city, and the retry logic lives in one place.
    '''
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': START, 'end_date': END,
        'hourly': 'temperature_2m',
        'temperature_unit': 'fahrenheit',
        'timezone': 'GMT',
    }
    for attempt in range(4):
        # WHY try/except: network hiccups are temporary and should trigger a retry, not a crash.
        try:
            resp = requests.get(ARCHIVE_URL, params=params, timeout=120)
            resp.raise_for_status()
            hourly = resp.json()['hourly']
            idx = pd.to_datetime(hourly['time'], utc=True)
            return pd.Series(hourly['temperature_2m'], index=idx, name=f'temp_{name}_f', dtype='float')
        except (requests.RequestException, KeyError, ValueError) as err:
            if attempt == 3:
                raise
            print(f'Retrying {name}: {err}')
            # WHY a long, growing wait: the free API rate-limits bursts (error 429).
            time.sleep(20 * (attempt + 1))

series = []
for name, c in CITIES.items():
    s = fetch_city(name, c['lat'], c['lon'])
    print(f'{name}: {s.notna().sum():,} hours')
    series.append(s)
    time.sleep(10)  # WHY: spacing requests avoids the free API's rate limit

weather = pd.concat(series, axis=1)
weather.index.name = 'period_utc'
weather.head()

## 4. Weather features

- **`temp_tx_weighted_f`:** population-weighted Texas temperature.
- **`cooling_degree_f` / `heating_degree_f`:** how far the weighted temperature sits above or below 65°F, the standard balance point utilities use. Demand rises on both sides of it, so splitting it into two features lets a model learn that a 100°F hour and a 30°F hour both push demand up.

**Why the weighted average skips missing cities:** if one city has a gap for an hour, averaging the remaining cities (re-weighted) is better than losing the whole hour.

In [ ]:
temp_cols = [f'temp_{n}_f' for n in CITIES]
w = pd.Series({f'temp_{n}_f': WEIGHTS[n] for n in CITIES})

# WHY multiply then divide by the weights present: re-normalizes when a city is missing that hour.
present = weather[temp_cols].notna()
weather['temp_tx_weighted_f'] = (
    (weather[temp_cols] * w).sum(axis=1, min_count=1) / (present * w).sum(axis=1).replace(0, np.nan)
)

BALANCE_F = 65
weather['cooling_degree_f'] = (weather['temp_tx_weighted_f'] - BALANCE_F).clip(lower=0)
weather['heating_degree_f'] = (BALANCE_F - weather['temp_tx_weighted_f']).clip(lower=0)

weather.to_csv('weather_texas_hourly.csv')
weather.describe().T.round(1)

## 5. Weather quality checks

In [ ]:
# WHY rebuild the full range: shows hours the archive left out entirely, not just nulls.
expected = pd.date_range(weather.index.min(), weather.index.max(), freq='h')
print('Rows:', len(weather), '| expected:', len(expected))
print('Missing hours:', len(expected.difference(weather.index)))
print('\nNulls per column:')
print(weather.isna().sum())

### Observations

*Write what the output shows here.*

## 6. Merge onto the ERCOT data

**Why a left join on the ERCOT index:** every ERCOT hour stays, even if weather were missing for it, so the demand series is never shortened.

**Why a new file:** `eia930_erco_cleaned.csv` stays untouched, so models already built on it don't break.

In [ ]:
erco = pd.read_csv('eia930_erco_cleaned.csv')
# WHY utc=True: matches the weather index exactly.
erco['period_utc'] = pd.to_datetime(erco['period_utc'], utc=True)
erco = erco.set_index('period_utc').sort_index()

merged = erco.join(weather, how='left')
print('ERCOT rows:', len(erco), '| merged rows:', len(merged))
print('Hours with no weighted temperature:', int(merged['temp_tx_weighted_f'].isna().sum()))

### Weather lag features

**Why:** like demand, a day-ahead model may use yesterday's temperature safely. Same-hour temperature is also kept; see the note below on how to treat it.

In [ ]:
merged['temp_tx_weighted_lag_24'] = merged['temp_tx_weighted_f'].shift(24)
merged.to_csv('eia930_erco_weather.csv')
merged[['demand_mwh', 'temp_tx_weighted_f', 'cooling_degree_f', 'heating_degree_f']].head()

## 7. Quick check: does temperature track demand?

In [ ]:
# WHY correlate against degree-hours, not raw temperature: demand rises on both hot and cold
# extremes, so raw temperature hides the winter effect.
merged[['demand_mwh', 'temp_tx_weighted_f', 'cooling_degree_f', 'heating_degree_f']].corr()['demand_mwh'].round(3)

## 8. Notes for the Data Scientist

- **Actual vs. forecast weather:** these are *observed* temperatures. In real day-ahead forecasting, the operator only has a *weather forecast* for tomorrow. Using observed same-hour temperature gives the model a "perfect weather forecast," so its accuracy is a best case. Either state that clearly in the report, or use only `temp_tx_weighted_lag_24`, which is fully known a day ahead.
- **Lag rows:** the first 24 rows have no weather lag, in addition to the 168 rows with no demand lag.
- **Source to cite:** Zippenfenig, P. (2023). *Open-Meteo.com Weather API* [Computer software]. Zenodo. https://doi.org/10.5281/ZENODO.7970649 — the historical data comes from the ERA5 reanalysis (Hersbach et al., 2020).

### Observations

*Write what the correlation and merge checks show here.*